##  ME 263: Applied FEM

### Lecture 7 (31/8/2026) —  How well does $u_h$ ∈ $V_h$ approximate u in V ?

### We have two problem to tackle 
- 1. How much accurate the solution (1% or 10 %) ?
- 2. How do we measure and which one is more relevent <br> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
                    -a) energy norm <br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
                    -b) $H^1$ norm


### Energy Norm ($\vert{}\vert{}\cdot\vert{}\vert{}_a$)
- The energy norm is the fundamental metric that the Galerkin method naturally minimizes.
- Instead of measuring the error in the primary physical variable itself ( displacement or temperature), it measures the error in its derivatives.
- Because the finite element solution $u_h$ is an orthogonal projection of the exact continuous solution $u$, the Galerkin method mathematically guarantees that $u_h$ is the absolute best possible approximation of $u$ that can exist within your chosen mesh subspace ($V_h$).

<img src="./image/galerkin_othogonality.jpg" style="display: block; margin: 0 auto;" width="400">

### best approximation 
$$\vert{}\vert{}u - v_h\vert{}\vert{}_a^2 = ? $$

proof

$$\vert{}\vert{}u- v_h\vert{}\vert{}_a^2 = a (u - v_h, u - v_h) $$

adding and subtract $u_h$ in RHS
$$\vert{}\vert{}u- v_h\vert{}\vert{}_a^2 = a (u - u_h + u_h -  v_h, u - v_h) $$

e = $u - u_h$
$$\vert{}\vert{}u- v_h\vert{}\vert{}_a^2 = a (e + u_h -  v_h, u - v_h) $$
$$ \vert{}\vert{}u- v_h\vert{}\vert{}_a^2 = a(e, u - v_h) + a(u_h - v_h, u - v_h)$$
substitution ($u - v_h = e + u_h - v_h$)
$$\vert{}\vert{}u- v_h\vert{}\vert{}_a^2= a(e, e + u_h - v_h) + a(u_h - v_h, e + u_h - v_h)$$
$$= a(e, e) + a(e, u_h - v_h) + a(u_h - v_h, e) + a(u_h - v_h, u_h - v_h)$$
$$= a(e, e) + 2a(e, u_h - v_h) + a(u_h - v_h, u_h - v_h)$$

using Galerkin Orthogonality  $$2a(e, u_h - v_h) = 0$$

The equation simplifies to a Pythagorean identity:

$$\vert{}\vert{}u - v_h\vert{}\vert{}_a^2 = \vert{}\vert{}e\vert{}\vert{}_a^2 + \vert{}\vert{}u_h - v_h\vert{}\vert{}_a^2$$


<img src="./image/pythagoras_triangle.jpg" style="display: block; margin: 0 auto;" width="400">

$$\vert{}\vert{}u_h - v_h\vert{}\vert{}_a^2 ≥ 0 $$

$$\vert{}\vert{} e \vert{}\vert{}_a^2 ≤  \vert{}\vert{}u - v_h\vert{}\vert{}_a^2 $$
$u_h$ is the best approximation u in $\vert{}\vert{} . \vert{}\vert{}_a$ <br>
$u_h$ is not nesnecessarily best in any other norm.

### approximation u by $u_h$ in the $H^1$ norm
Céa's Lemma :
- a(.,.) is bounded in $H^1$ norm 
    - $\vert{}a(v,w)\vert{} \leq C_1 \vert{}\vert{}v\vert{}\vert{}_{H^1} \vert{}\vert{}w\vert{}\vert{}_{H^1}$ some constant $C_1$ > 0 and all v,w in V

- L is bounded in the $H^1$ norm
    - $\vert{}L(v)\vert{} \leq C_2 \vert{}\vert{}v\vert{}\vert{}_{H^1}$ some constant $C_2$ > 0 and all v in V
- a is Coercive in the $H^1$ norm
    - $a(v, v) \geq \alpha \vert{}\vert{}v\vert{}\vert{}_{H^1}^2$

<br>



$$a(u - u_h, u - u_h) = a(u - u_h, u - v_h + v_h - u_h)$$
Apply Linearity and Apply Galerkin Orthogonality
$$a(u - u_h, u - u_h) = a(u - u_h, u - v_h)$$

from Céa's Lemma : 1st conditon a(.,.) is bounded 
$$a(u - u_h, u - v_h) \leq C_1 \vert{}\vert{}u - u_h\vert{}\vert{}_{H^1} \vert{}\vert{}u - v_h\vert{}\vert{}_{H^1}$$

from Coercive 
$$C_2 \vert{}\vert{}u - u_h\vert{}\vert{}_{H^1}^2 \leq a(u - u_h, u - u_h)$$

$$C_2 \vert{}\vert{}u - u_h\vert{}\vert{}_{H^1}^2 \leq C_1 \vert{}\vert{}u - u_h\vert{}\vert{}_{H^1} \vert{}\vert{}u - v_h\vert{}\vert{}_{H^1}$$
$$\vert{}\vert{}u - u_h\vert{}\vert{}_{H^1} \leq \frac{C_1}{C_2} \vert{}\vert{}u - v_h\vert{}\vert{}_{H^1}$$

<br><br>
$$\vert{}\vert{}u - u_h\vert{}\vert{}_{H^1} \leq \frac{C_1}{C_2} \inf_{v_h \in V_h} \vert{}\vert{}u - v_h\vert{}\vert{}_{H^1}$$

 the $H^1$ norm to the energy norm by factors dependent on the constants $C_1$ and $C_2$

## code part

In [ ]:
def energy_norm(e,msh):
    s = fem.assemble_scalar(fem.form(ufl.inner(ufl.grad(e),ufl.grad(e))*ufl.dx(metadata={"quadrature_degree":8})))
    return np.sqrt(msh.comm.allreduce(s, op=MPI.SUM))

def L2_norm(e,msh):
    s = fem.assemble_scalar(fem.form(e*e*ufl.dx(metadata={"quadrature_degree":8})))
    return np.sqrt(msh.comm.allreduce(s,op=MPI.SUM))

def galerkin_errors(msh,V,ue):
    facets = mesh.locate_entities_boundary(
        msh,msh.topology.dim - 1,
        marker=lambda x: np.full(x.shape[1],True))
    dofs = fem.locate_dofs_topological(V,msh.topology.dim-1,facets)
    bc = fem.dirichletbc(0.0,dofs, V)

    u = ufl.TrialFunction(V)
    v = ufl.TestFunction(V)
    x = ufl.SpatialCoordinate(msh)
    f = 2 * ufl.pi**2 * ufl.sin(ufl.pi*x[0]) * ufl.sin(ufl.pi*x[1])
    a = ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
    L = f*v*ufl.dx

    problem = LinearProblem(
        a,L, bcs=[bc],
        petsc_options_prefix="poisson_",
        petsc_options={"ksp_type": "preonly", "pc_type": "lu"}
    )

    uh = problem.solve()

    e = uh - ue
    return energy_norm(e,msh),L2_norm(e,msh)

def interpolation_errors(msh,V,ue):
    ui = fem.Function(V)
    ui.interpolate(lambda x: np.sin(np.pi*x[0])*np.sin(np.pi*x[1]))
    e = ue - ui
    return energy_norm(e,msh), L2_norm(e,msh)

def L2projection_errors(msh,V,ue):
    u = ufl.TrialFunction(V)
    v = ufl.TestFunction(V)
    a = u*v*ufl.dx
    L = ue*v*ufl.dx(metadata={"quadrature_degree":8})
    uL2 = LinearProblem(a,L,bcs=[],
                        petsc_options_prefix="L2_",
                        petsc_options={"ksp_type": "preonly", "pc_type": "lu"
                    }).solve()
    e = ue - uL2
    return energy_norm(e,msh), L2_norm(e,msh)

Ns = [8,16,32,64,128]
hs = np.array([1.0/N for N in Ns])

gal_en,gal_L2 = [],[]
interp_en, interp_L2 = [],[]
proj_en, proj_L2 = [],[]

for N in Ns:
    print("Mesh density: ", N)
    msh = mesh.create_unit_square(MPI.COMM_WORLD,N,N)
    V = fem.functionspace(msh,("Lagrange",1))
    x = ufl.SpatialCoordinate(msh)
    ue = ufl.sin(ufl.pi*x[0])*ufl.sin(ufl.pi*x[1])
    a,b = galerkin_errors(msh,V,ue); gal_en.append(a); gal_L2.append(b)
    a,b = interpolation_errors(msh,V,ue); interp_en.append(a);interp_L2.append(b)
    a,b = L2projection_errors(msh,V,ue); proj_en.append(a); proj_L2.append(b)

_,ax = plt.subplots(1,2)

ax[0].loglog(hs, gal_en, "o-", label="uh")
ax[0].loglog(hs, interp_en, "s-", label="ui")
ax[0].loglog(hs, proj_en, "^-", label="u2")
ax[0].legend()

ax[1].loglog(hs, gal_L2, "o-", label="uh")
ax[1].loglog(hs, interp_L2, "s-", label="ui")
ax[1].loglog(hs, proj_L2, "^-", label="u2")
ax[1].legend()

plt.show()

: 

comparision between energy norm and L2 norm. 
<img src="./image/energy_norm_vs_L2_norm.jpg" style="display: block; margin: 0 auto;" width="400">
